# DataMate: Qwen3.5-9B bản đầy đủ trên vLLM (2 x T4, fp16)

Trước khi chạy: **Settings → Accelerator: GPU T4 x2**, **Internet: On**.
Chạy lần lượt từng ô. Ô **5** in ra **1 link + KEY**: gửi lại cả hai. Giữ tab mở, ô cuối chạy mãi để phiên không tắt.

- Model gốc `Qwen/Qwen3.5-9B` (khoảng 19 GB), chia đôi qua 2 card (`--tensor-parallel-size 2`).
- T4 không có bf16, nên chạy **fp16** (`--dtype half`). Đây là bản khác với Colab bf16: chỉ dùng cho phép so kín (mốc và bản thử cùng chạy trên server này).
- vLLM chạy song song nhiều request, khác ollama qwen35 chỉ chạy từng request một.
- Đây là **lần thử**: ô 4 thử lần lượt 3 cấu hình. Ô 5 kiểm câu trả lời có bình thường không (fp16 có thể tràn số) và đo tốc độ 16 luồng. Có ô nào báo lỗi thì chép phần log in ra gửi lại.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -n 2
!python -c "import torch, sys; print('python', sys.version.split()[0], 'torch', torch.__version__)"

In [ ]:
%%bash
# cài vLLM 0.17.0 (cùng bản với Colab) + cloudflared; mất khoảng 5-10 phút
set -e
pip install -q "vllm==0.17.0" "huggingface_hub[hf_transfer]" > /tmp/pip_vllm.log 2>&1 || (tail -n 30 /tmp/pip_vllm.log; exit 1)
python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__)"
wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
chmod +x /usr/local/bin/cloudflared
cloudflared --version

In [ ]:
# tải model về /tmp (không dùng /kaggle/working vì chỉ có khoảng 20 GB)
import os, subprocess
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
M = "/tmp/qwen35_9b"
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen3.5-9B", local_dir=M, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja", "*.py"])
print(subprocess.run(["du", "-sh", M], capture_output=True, text=True).stdout)
print(sorted(os.listdir(M)))

In [ ]:
# bật vLLM: thử lần lượt 3 cấu hình, dừng ở cấu hình đầu tiên chạy được
import os, glob, subprocess, time, json, secrets, urllib.request
D = "/tmp/datamate"; os.makedirs(D, exist_ok=True)
# FlashInfer biên dịch kernel cho T4 lúc chạy và liên kết -lcuda. Kaggle chỉ có libcuda.so.1 (ở /usr/local/nvidia/lib64),
# thư mục CUDA chỉ đọc -> tạo libcuda.so trong /tmp và chỉ trình liên kết tìm ở đó (vLLM thừa hưởng biến môi trường)
L = "/tmp/libcuda_link"; os.makedirs(L, exist_ok=True)
SRC = sorted(glob.glob("/usr/local/nvidia/lib64/libcuda.so.1") + glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so.1")
             + glob.glob("/usr/lib64/libcuda.so.1"))
if not os.path.lexists(f"{L}/libcuda.so"):
    os.symlink(SRC[0], f"{L}/libcuda.so")
os.environ["LIBRARY_PATH"] = L + ":" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = os.path.dirname(SRC[0]) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
subprocess.run("rm -rf /root/.cache/flashinfer", shell=True)
open("/tmp/t.cpp", "w").write("int main(){return 0;}")
_r = subprocess.run("c++ /tmp/t.cpp -lcuda -o /tmp/t", shell=True, capture_output=True, text=True)
print("liên kết -lcuda:", "OK" if _r.returncode == 0 else "LỖI " + _r.stderr[-300:], "| driver:", SRC[0])
KEY = ""   # để trống: tạo KEY ngẫu nhiên. Acc thứ hai: dán KEY của acc thứ nhất vào đây để mọi server dùng chung một KEY
KEY = KEY.strip() or secrets.token_hex(8)
PORT = 8000
CHUNG = ["vllm", "serve", M, "--served-model-name", "qwen", "--port", str(PORT), "--api-key", KEY,
         "--dtype", "half", "--tensor-parallel-size", "2", "--seed", "2026", "--max-num-seqs", "32",
         "--no-enable-prefix-caching"]
MM0 = ["--limit-mm-per-prompt", json.dumps({"image": 0, "video": 0})]
CAU_HINH = [
    ("nhanh",   ["--max-model-len", "16384", "--gpu-memory-utilization", "0.92"] + MM0),
    ("an_toan", ["--max-model-len", "16384", "--gpu-memory-utilization", "0.88", "--enforce-eager"] + MM0),
    ("toi_gian", ["--max-model-len", "12288", "--gpu-memory-utilization", "0.88", "--enforce-eager"]),
]

def song():
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/models", headers={"Authorization": f"Bearer {KEY}"})
        urllib.request.urlopen(req, timeout=5); return True
    except Exception:
        return False

def tra_loi_duoc():
    # /v1/models trả lời được chưa đủ: kernel attention chỉ biên dịch ở request đầu tiên -> phải hỏi thật một câu
    body = {"model": "qwen", "temperature": 0, "max_tokens": 5, "messages": [{"role": "user", "content": "1+1=?"}],
            "chat_template_kwargs": {"enable_thinking": False}}
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/chat/completions", data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json", "Authorization": f"Bearer {KEY}"})
        urllib.request.urlopen(req, timeout=900).read(); time.sleep(5); return song()
    except Exception:
        return False

proc = None
for ten, co in CAU_HINH:
    log = f"{D}/vllm_{ten}.log"
    print("== thử", ten, time.strftime("%H:%M:%S"), flush=True)
    proc = subprocess.Popen(CHUNG + co, stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(300):                      # tối đa 25 phút (lần đầu biên dịch kernel lâu)
        time.sleep(5)
        if song() or proc.poll() is not None:
            break
    if song() and tra_loi_duoc():
        print("== CHẠY ĐƯỢC với cấu hình", ten, time.strftime("%H:%M:%S")); CAU = ten; break
    print("   không lên. Các dòng lỗi:")
    print(subprocess.run(f"grep -nE 'Error|error|out of memory|cannot find|not supported|Killed' {log} | grep -v Traceback | tail -15 | cut -c1-400",
                         shell=True, capture_output=True, text=True).stdout)
    proc.kill(); proc.wait(); time.sleep(10)
else:
    raise SystemExit("cả 3 cấu hình đều không chạy được: gửi lại phần log ở trên")
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

In [ ]:
# kiểm câu trả lời (fp16 có tràn số không) + đo tốc độ 16 luồng, rồi mở đường hầm
import re, threading
def hoi(noi_dung, max_tokens=200, temperature=0):
    body = {"model": "qwen", "temperature": temperature, "max_tokens": max_tokens,
            "messages": [{"role": "system", "content": "Bạn là trợ lý phân tích dữ liệu cửa hàng. Trả lời ngắn gọn."},
                         {"role": "user", "content": noi_dung}],
            "chat_template_kwargs": {"enable_thinking": False}}
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json", "Authorization": f"Bearer {KEY}"})
    return json.loads(urllib.request.urlopen(req, timeout=600).read())

for q in ["1+1 bằng mấy? Chỉ trả lời con số.",
          "Viết câu SQL DuckDB tính tổng cột thanh_tien của bảng don_hang trong tháng 5/2025 (cột ngay_dat kiểu DATE). Chỉ trả SQL.",
          "Liệt kê 5 tỉnh thành Việt Nam, mỗi dòng một tên."]:
    t = time.time(); r = hoi(q)
    print(f"{time.time() - t:.1f}s", repr(r["choices"][0]["message"]["content"][:300]))

# 16 luồng, prompt dài cỡ thật (khoảng 3000 token)
dai = "Bảng don_hang(ma_don, ngay_dat, ma_khach, thanh_tien, trang_thai). " * 150 + "\nCó bao nhiêu cột trong bảng don_hang? Chỉ trả con số."
kq = []
def chay():
    t = time.time(); r = hoi(dai, max_tokens=150, temperature=0.5); kq.append((time.time() - t, r["usage"]["completion_tokens"], r["choices"][0]["message"]["content"][:40]))
t0 = time.time(); th = [threading.Thread(target=chay) for _ in range(16)]
[x.start() for x in th]; [x.join() for x in th]
tong = time.time() - t0
print(f"16 luồng: {tong:.1f}s tổng, {sum(k[1] for k in kq) / tong:.1f} token sinh/giây, chậm nhất {max(k[0] for k in kq):.1f}s")
print("mẫu:", [k[2] for k in kq[:4]])

log = f"{D}/tunnel.log"
subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                 stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
url = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log).read())
    if m: url = m.group(0); break
print("\nGỬI LẠI 2 DÒNG NÀY (cấu hình:", CAU + "):")
print("URL =", url)
print("KEY =", KEY)

In [ ]:
# giữ phiên chạy; mỗi 5 phút in trạng thái (None = server còn sống)
while True:
    time.sleep(300)
    print(time.strftime("%H:%M"), proc.poll(), "song" if song() else "CHẾT", flush=True)